# CRIS Tasks v77 - scheduled refresh of the six CRIS tables

Six Snowflake tasks, one per table, chained in build order. Only the root task has a schedule; each of the others runs `AFTER` the one before it, so the chain is one run.

| Task | Table | Runs after |
|---|---|---|
| T1_TELECAST_FACT | CRIS_TELECAST_FACT | schedule (root) |
| T2_HALFHR_FACT | CRIS_HALFHR_FACT | T1 |
| T3_QH_FACT | CRIS_QH_FACT | T2 |
| T4_LATEST_DATE | CRIS_LATEST_DATE | T3 |
| T5_NETWORK_UNIVERSE | CRIS_NETWORK_UNIVERSE | T4 |
| T6_FY_CALENDAR | CRIS_FY_CALENDAR | T5 |

**Build-and-swap.** Each task builds `<table>_NEW`, swaps it in with `ALTER TABLE ... SWAP WITH`, and drops the old copy. The live table is never empty while it rebuilds, so a CRIS question during the refresh still gets an answer (from the previous day's data).

**Schedule.** The root fires every two hours through the night, 21:00, 23:00, 01:00, 03:00, 05:00 and 07:00 America/New_York, and rebuilds only if the source view has a newer BROADCAST_DATE than CRIS_TELECAST_FACT; otherwise it logs SKIPPED and the children do not run. Whichever firing comes after A&E's load picks it up, before the team sits down at 9:00. Once the load time is known, reduce it to one firing (the CRON in T1 only).

**Log.** Every run writes to `CRIS_REFRESH_LOG`: one SKIPPED row when there is nothing new, or one REBUILT row per table with its row count and latest BROADCAST_DATE. Cell 0 creates the log table; cell 11 reads it.

**Role.** EXT_SIIRA_DEV_SVC_ROLE has CREATE TASK on the schema, EXECUTE TASK on the account and USAGE/OPERATE on EXT_SIIRA_XS_WH. The warehouse is an XS; the telecast and quarter-hour builds took several minutes on it by hand, so the first scheduled run should be watched in TASK_HISTORY.

**The SQL is the v77 table build, unchanged**, only wrapped: the half-hour history window (3 years) is inlined because session variables do not exist inside a task.

Run cells 1-6 to create the tasks, cell 7 to enable them (children first, root last), cell 8 to run the chain once now, cell 9 to watch it.


## 0. The refresh log (create once)

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (
    RUN_TS             TIMESTAMP_LTZ,
    TABLE_NAME         VARCHAR,
    STATUS             VARCHAR,       -- REBUILT | SKIPPED
    ROW_COUNT          NUMBER,
    MAX_BROADCAST_DATE DATE,
    NOTE               VARCHAR
);

## 1. T1_TELECAST_FACT -> CRIS_TELECAST_FACT

In [ ]:
%%sql
CREATE OR REPLACE TASK AUDIENCE_DEV_DB.CRIS.T1_TELECAST_FACT
  WAREHOUSE = EXT_SIIRA_XS_WH
  SCHEDULE = 'USING CRON 0 21,23,1,3,5,7 * * * America/New_York'
AS
BEGIN
  LET src_max DATE := (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DB.UBD.VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED);
  LET tbl_max DATE := (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT);
  IF (:src_max <= :tbl_max) THEN
    INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
      VALUES (CURRENT_TIMESTAMP(), 'CRIS_TELECAST_FACT', 'SKIPPED', NULL, :tbl_max, 'No new data: source ' || :src_max);
    RETURN 'No new data: source ' || :src_max || ', table ' || :tbl_max;
  END IF;
  CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT_NEW
CLUSTER BY (BROADCAST_DATE, NETWORK_CODE)
AS
WITH src AS (
    SELECT
        NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON, IS_PREMIERE_CT,
        CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
        PROG_TYPE_ALPHA,
        START_TIME, END_TIME, EVENT_DURATION, CONTRIBUTING_DURATION,
        COMMERCIAL_DURATION,
        EVENT_START_HR, EVENT_START_MIN,
        REPEAT_IND, MOVIE_IND, IS_SPECIAL, SHORT_DURATION_IND,
        PROG_PREMIERE_IND, PROG_ORIG_ACQUIRED_CD, EPISODE_ORIG_ACQUIRED_CD,
        MEDIAN_AGE,
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_17_AA_ESTIMATES, F2_5_AA_ESTIMATES, F2_AA_ESTIMATES,
    F30_34_AA_ESTIMATES, F35_39_AA_ESTIMATES, F35_49_AA_ESTIMATES,
    F35_54_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50PLUS_AA_ESTIMATES, F50_54_AA_ESTIMATES, F50_64_AA_ESTIMATES,
    F55PLUS_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_17_AA_ESTIMATES, M2_5_AA_ESTIMATES, M2_AA_ESTIMATES,
    M30_34_AA_ESTIMATES, M35_39_AA_ESTIMATES, M35_49_AA_ESTIMATES,
    M35_54_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50PLUS_AA_ESTIMATES, M50_54_AA_ESTIMATES, M50_64_AA_ESTIMATES,
    M55PLUS_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_49_AA_ESTIMATES,
    P35_54_AA_ESTIMATES, P50_64_AA_ESTIMATES, P50_AA_ESTIMATES,
    P55_AA_ESTIMATES, P65_AA_ESTIMATES,
        MEDIA_TYPE_CODE, ORIGINATOR_ID,
        TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
        FILE_NAME, FILE_DATE,
        IFF(UPPER(COALESCE(FILE_NAME,'')) LIKE 'UBD%','BIGDATA','PANEL')  AS SOURCE_FEED,
        NULLIF(TRIM(COALESCE(VIEWING_TYPE,'')),'')                        AS VT
    FROM AUDIENCE_DB.UBD.VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED
    WHERE PROGRAM_PUT_INDICATOR = 0
      AND TOTAL_PROGRAM_IND     = '00'
),
ranked AS (
    SELECT src.*,
        IFF(SOURCE_FEED = 'BIGDATA', 0, 1)                                AS FEED_RANK,
        IFF(REGEXP_LIKE(UPPER(COALESCE(FILE_NAME,'')), '.*_CW.*'), 0, 1)  AS WKLY_RANK,
        IFF(VT IN ('1','3','5','B'), 1, 0)                                AS VT_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(TRACKAGE_ID)),0) = 0, 0, 1) AS TRK_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(MARKET_BREAK_ID)),0) = 0, 0, 1) AS MKT_RANK
    FROM src
)
SELECT
    NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
    PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
    PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
    START_TIME, END_TIME, EVENT_DURATION, CONTRIBUTING_DURATION,
    COMMERCIAL_DURATION,
    (MOD(MOD(EVENT_START_HR, 24) + 18, 24) + 6) * 100 + EVENT_START_MIN  AS BCAST_START_2959,
    IFF(REPEAT_IND = 0, TRUE, FALSE)                                AS IS_FIRST_RUN_CT,
    IFF(CT_SEASON IS NOT NULL, TRUE, FALSE)                         AS IS_FIRST_RUN_CABLE_TRACKS,
    IS_PREMIERE_CT                                                  AS IS_PREMIERE_CT_SOURCE,
    IFF(CT_SEASON IS NOT NULL
        AND TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
            || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0')
          = MIN(TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
                || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0'))
            OVER (PARTITION BY NETWORK_CODE, RATING_SOURCE,
                               PROGRAM_CODE, CT_SEASON),
        1, 0)                                                       AS IS_PREMIERE_CT,
    REPEAT_IND, MOVIE_IND, IS_SPECIAL, SHORT_DURATION_IND,
    CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE, PROG_TYPE_ALPHA,
    IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PROG_PREMIERE_IND)),0) = 1, 1, 0) AS IS_SEASON_PREMIERE,
    CASE WHEN MOVIE_IND = 1 THEN 'Movie'
         WHEN UPPER(COALESCE(CT_GENRE_1,'')) = 'SPECIAL' OR IS_SPECIAL = 1 THEN 'Special'
         ELSE 'Series' END                                               AS CONTENT_TYPE,
    CASE WHEN MOVIE_IND = 1
           AND ( UPPER(PROGRAM_NAME) LIKE '% HOL %'
              OR UPPER(PROGRAM_NAME) LIKE '%HOLIDAY%' )
         THEN TRUE ELSE FALSE END                                        AS IS_HOLIDAY,
    COALESCE(NULLIF(TRIM(COALESCE(EPISODE_ORIG_ACQUIRED_CD,'')),''),
             NULLIF(TRIM(COALESCE(PROG_ORIG_ACQUIRED_CD,'')),''))        AS ORIG_ACQUIRED_CD,
    CASE WHEN UPPER(PROGRAM_NAME) LIKE '%ORIGINAL%' THEN 'ORIGINAL'
         WHEN UPPER(PROGRAM_NAME) LIKE '%ACQUIRED%' THEN 'ACQUIRED'
         WHEN UPPER(PROGRAM_NAME) LIKE '%FEATURE%'  THEN 'FEATURE'
         ELSE NULL END                                                   AS MOVIE_TYPE,
    MEDIAN_AGE,
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_17_AA_ESTIMATES, F2_5_AA_ESTIMATES, F2_AA_ESTIMATES,
    F30_34_AA_ESTIMATES, F35_39_AA_ESTIMATES, F35_49_AA_ESTIMATES,
    F35_54_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50PLUS_AA_ESTIMATES, F50_54_AA_ESTIMATES, F50_64_AA_ESTIMATES,
    F55PLUS_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_17_AA_ESTIMATES, M2_5_AA_ESTIMATES, M2_AA_ESTIMATES,
    M30_34_AA_ESTIMATES, M35_39_AA_ESTIMATES, M35_49_AA_ESTIMATES,
    M35_54_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50PLUS_AA_ESTIMATES, M50_54_AA_ESTIMATES, M50_64_AA_ESTIMATES,
    M55PLUS_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_49_AA_ESTIMATES,
    P35_54_AA_ESTIMATES, P50_64_AA_ESTIMATES, P50_AA_ESTIMATES,
    P55_AA_ESTIMATES, P65_AA_ESTIMATES,
    CASE NETWORK_CODE
        WHEN 'LIF'  THEN F25_64_AA_ESTIMATES
        WHEN 'LMN'  THEN F25_64_AA_ESTIMATES
        WHEN 'VICE' THEN P18_49_AA_ESTIMATES
        ELSE             P25_64_AA_ESTIMATES
    END                                                                  AS DEMO_AA,
    CASE NETWORK_CODE WHEN 'HIST' THEN M2_AA_ESTIMATES
                      ELSE             F2_AA_ESTIMATES END               AS SKEW_AA,
    CASE NETWORK_CODE WHEN 'HIST' THEN 'MALE' ELSE 'FEMALE' END          AS SKEW_DIRECTION,
    MEDIA_TYPE_CODE, ORIGINATOR_ID,
    SOURCE_FEED, VT AS VIEWING_TYPE,
    TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
    FILE_NAME, FILE_DATE,
    CURRENT_TIMESTAMP()                                                  AS LOADED_AT
FROM ranked
QUALIFY ROW_NUMBER() OVER (
        PARTITION BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                     PROGRAM_CODE, TELECAST_NUM
        ORDER BY FEED_RANK, WKLY_RANK, VT_RANK, TRK_RANK, MKT_RANK,
                 FILE_DATE DESC NULLS LAST, FILE_NAME DESC NULLS LAST
    ) = 1;
  ALTER TABLE AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT SWAP WITH AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT_NEW;
  DROP TABLE IF EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT_NEW;
  INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
    SELECT CURRENT_TIMESTAMP(), 'CRIS_TELECAST_FACT', 'REBUILT', COUNT(*), MAX(BROADCAST_DATE), NULL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT;
END;

## 2. T2_HALFHR_FACT -> CRIS_HALFHR_FACT

In [ ]:
%%sql
CREATE OR REPLACE TASK AUDIENCE_DEV_DB.CRIS.T2_HALFHR_FACT
  WAREHOUSE = EXT_SIIRA_XS_WH
  AFTER AUDIENCE_DEV_DB.CRIS.T1_TELECAST_FACT
AS
BEGIN
  CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT_NEW
CLUSTER BY (BROADCAST_DATE, NETWORK_CODE)
AS
WITH src AS (
    SELECT
        NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON, IS_PREMIERE_CT,
        CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
        PROG_TYPE_ALPHA,
        START_TIME, END_TIME, EVENT_START_HR, EVENT_START_MIN,
        HALFHR_START_TIME, HH_EVENT_START_HR, HH_EVENT_START_MIN,
        CONTRIBUTING_DURATION, COMMERCIAL_DURATION,
    CASE WHEN MOVIE_IND = 1
           AND ( UPPER(PROGRAM_NAME) LIKE '% HOL %'
              OR UPPER(PROGRAM_NAME) LIKE '%HOLIDAY%' )
         THEN TRUE ELSE FALSE END                                        AS IS_HOLIDAY,
        REPEAT_IND, MOVIE_IND, IS_SPECIAL, PROG_PREMIERE_IND,
        MEDIAN_AGE,
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
    F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
    M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
    P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
        MEDIA_TYPE_CODE, ORIGINATOR_ID,
        TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
        FILE_NAME, FILE_DATE,
        IFF(UPPER(COALESCE(FILE_NAME,'')) LIKE 'UBD%','BIGDATA','PANEL')  AS SOURCE_FEED,
        NULLIF(TRIM(COALESCE(VIEWING_TYPE,'')),'')                        AS VT
    FROM AUDIENCE_DB.UBD.VW_HALFHR_PROGRAM_ESTIMATES_UNIFIED
    WHERE PROGRAM_PUT_INDICATOR = 0
      AND CONTRIBUTING_DURATION > 0
      AND BROADCAST_DATE >= DATEADD('year', -3, CURRENT_DATE())
),
ranked AS (
    SELECT src.*,
        IFF(SOURCE_FEED = 'BIGDATA', 0, 1)                                AS FEED_RANK,
        IFF(REGEXP_LIKE(UPPER(COALESCE(FILE_NAME,'')), '.*_CW.*'), 0, 1)  AS WKLY_RANK,
        IFF(VT IN ('1','3','5','B'), 1, 0)                                AS VT_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(TRACKAGE_ID)),0) = 0, 0, 1) AS TRK_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(MARKET_BREAK_ID)),0) = 0, 0, 1) AS MKT_RANK
    FROM src
)
SELECT
    NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
    PROGRAM_CODE, TELECAST_NUM,
    PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
    CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
    PROG_TYPE_ALPHA,
    START_TIME, END_TIME,
    HALFHR_START_TIME,
    (MOD(MOD(HH_EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + HH_EVENT_START_MIN                                       AS HH_START_2959,
    (MOD(MOD(EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + EVENT_START_MIN                                          AS EP_START_2959,
    CONTRIBUTING_DURATION                                        AS HH_MIN,
    COMMERCIAL_DURATION,
    IFF(REPEAT_IND = 0, TRUE, FALSE)                                AS IS_FIRST_RUN_CT,
    IFF(CT_SEASON IS NOT NULL, TRUE, FALSE)                         AS IS_FIRST_RUN_CABLE_TRACKS,
    IS_PREMIERE_CT                                                  AS IS_PREMIERE_CT_SOURCE,
    IFF(CT_SEASON IS NOT NULL
        AND TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
            || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0')
          = MIN(TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
                || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0'))
            OVER (PARTITION BY NETWORK_CODE, RATING_SOURCE,
                               PROGRAM_CODE, CT_SEASON),
        1, 0)                                                       AS IS_PREMIERE_CT,
    REPEAT_IND, MOVIE_IND, IS_SPECIAL,
    IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PROG_PREMIERE_IND)),0) = 1, 1, 0) AS IS_SEASON_PREMIERE,
    CASE WHEN MOVIE_IND = 1 THEN 'Movie'
         WHEN UPPER(COALESCE(CT_GENRE_1,'')) = 'SPECIAL' OR IS_SPECIAL = 1 THEN 'Special'
         ELSE 'Series' END                                       AS CONTENT_TYPE,
    MEDIAN_AGE,
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
    F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
    M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
    P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
    CASE NETWORK_CODE
        WHEN 'LIF'  THEN F25_64_AA_ESTIMATES
        WHEN 'LMN'  THEN F25_64_AA_ESTIMATES
        WHEN 'VICE' THEN P18_49_AA_ESTIMATES
        ELSE             P25_64_AA_ESTIMATES
    END                                                          AS DEMO_AA,
    CASE NETWORK_CODE WHEN 'HIST' THEN M2_AA_ESTIMATES
                      ELSE             F2_AA_ESTIMATES END       AS SKEW_AA,
    MEDIA_TYPE_CODE, ORIGINATOR_ID,
    IFF(NETWORK_CODE IN ('AEN','HIST','LIF','LMN','FYI','VICE'), TRUE, FALSE)           AS IS_AE_NETWORK,
    SOURCE_FEED, VT AS VIEWING_TYPE,
    FILE_NAME, FILE_DATE,
    CURRENT_TIMESTAMP()                                          AS LOADED_AT
FROM ranked
QUALIFY ROW_NUMBER() OVER (
        PARTITION BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                     PROGRAM_CODE, TELECAST_NUM, HALFHR_START_TIME
        ORDER BY FEED_RANK, WKLY_RANK, VT_RANK, TRK_RANK, MKT_RANK,
                 FILE_DATE DESC NULLS LAST, FILE_NAME DESC NULLS LAST
    ) = 1;
  ALTER TABLE AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT SWAP WITH AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT_NEW;
  DROP TABLE IF EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT_NEW;
  INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
    SELECT CURRENT_TIMESTAMP(), 'CRIS_HALFHR_FACT', 'REBUILT', COUNT(*), MAX(BROADCAST_DATE), NULL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT;
END;

## 3. T3_QH_FACT -> CRIS_QH_FACT

In [ ]:
%%sql
CREATE OR REPLACE TASK AUDIENCE_DEV_DB.CRIS.T3_QH_FACT
  WAREHOUSE = EXT_SIIRA_XS_WH
  AFTER AUDIENCE_DEV_DB.CRIS.T2_HALFHR_FACT
AS
BEGIN
  CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT_NEW
CLUSTER BY (BROADCAST_DATE, NETWORK_CODE)
AS
WITH src AS (
    SELECT
        NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON, IS_PREMIERE_CT,
        CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE,
        PROG_TYPE_ALPHA,
        START_TIME, END_TIME, EVENT_START_HR, EVENT_START_MIN,
        QTR_START_TIME, QTR_EVENT_START_HR, QTR_EVENT_START_MIN,
        CONTRIBUTING_DURATION,
        REPEAT_IND, MOVIE_IND, IS_SPECIAL, PROG_PREMIERE_IND,
        F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
        F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
        F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
        F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
        F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
        F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
        F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
        F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
        M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
        M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
        M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
        M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
        M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
        M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
        M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
        M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
        P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
        P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
        P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
        P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
        TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
        FILE_NAME, FILE_DATE,
    IFF(UPPER(COALESCE(FILE_NAME,'')) LIKE 'UBD%','BIGDATA','PANEL')  AS SOURCE_FEED,
    NULLIF(TRIM(COALESCE(VIEWING_TYPE,'')),'')                        AS VT
    FROM AUDIENCE_DB.UBD.VW_QTRHR_PROGRAM_ESTIMATES_UNIFIED
    WHERE PROGRAM_PUT_INDICATOR = 0
      AND CONTRIBUTING_DURATION > 0
),
ranked AS (
    SELECT src.*,
        IFF(SOURCE_FEED = 'BIGDATA', 0, 1)                            AS FEED_RANK,
        IFF(VT IN ('1','3','5','B'), 1, 0)                            AS VT_RANK,
        IFF(REGEXP_LIKE(UPPER(COALESCE(FILE_NAME,'')), '.*_CW.*'), 0, 1) AS WKLY_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(TRACKAGE_ID)),0) = 0, 0, 1) AS TRK_RANK,
        IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(MARKET_BREAK_ID)),0) = 0, 0, 1) AS MKT_RANK
    FROM src
)
SELECT
    NETWORK_CODE, RATING_SOURCE,
    BROADCAST_DATE, DAY_OF_WEEK,
    PROGRAM_CODE, TELECAST_NUM, EPISODE_ID,
    PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
    START_TIME, END_TIME,
    QTR_START_TIME,
    CONTRIBUTING_DURATION                                      AS QH_MIN,
    (MOD(MOD(QTR_EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + QTR_EVENT_START_MIN                                     AS QH_START_2959,
    (MOD(MOD(EVENT_START_HR, 24) + 18, 24) + 6) * 100
      + EVENT_START_MIN                                         AS EP_START_2959,
    IFF(REPEAT_IND = 0, TRUE, FALSE)                                AS IS_FIRST_RUN_CT,
    IFF(CT_SEASON IS NOT NULL, TRUE, FALSE)                         AS IS_FIRST_RUN_CABLE_TRACKS,
    IS_PREMIERE_CT                                                  AS IS_PREMIERE_CT_SOURCE,
    IFF(CT_SEASON IS NOT NULL
        AND TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
            || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0')
          = MIN(TO_VARCHAR(BROADCAST_DATE, 'YYYYMMDD')
                || LPAD(TO_VARCHAR(TELECAST_NUM), 12, '0'))
            OVER (PARTITION BY NETWORK_CODE, RATING_SOURCE,
                               PROGRAM_CODE, CT_SEASON),
        1, 0)                                                       AS IS_PREMIERE_CT,
    REPEAT_IND, MOVIE_IND, IS_SPECIAL,
    CT_GENRE_1, CT_GENRE_2, CT_GENRE_3, CT_COMBINED_GENRE, PROG_TYPE_ALPHA,
    IFF(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(PROG_PREMIERE_IND)),0) = 1, 1, 0)
                                                                AS IS_SEASON_PREMIERE,
    CASE WHEN MOVIE_IND = 1 THEN 'Movie'
         WHEN UPPER(COALESCE(CT_GENRE_1,'')) = 'SPECIAL' OR IS_SPECIAL = 1 THEN 'Special'
         ELSE 'Series' END                                      AS CONTENT_TYPE,
    F12_14_AA_ESTIMATES, F15_17_AA_ESTIMATES, F18_20_AA_ESTIMATES,
    F18_34_AA_ESTIMATES, F18_49_AA_ESTIMATES, F18_AA_ESTIMATES,
    F21_24_AA_ESTIMATES, F21_AA_ESTIMATES, F25_29_AA_ESTIMATES,
    F25_54_AA_ESTIMATES, F25_64_AA_ESTIMATES, F25_AA_ESTIMATES,
    F2_5_AA_ESTIMATES, F2_AA_ESTIMATES, F30_34_AA_ESTIMATES,
    F35_39_AA_ESTIMATES, F40_44_AA_ESTIMATES, F45_49_AA_ESTIMATES,
    F50_54_AA_ESTIMATES, F55_64_AA_ESTIMATES, F65_AA_ESTIMATES,
    F6_8_AA_ESTIMATES, F9_11_AA_ESTIMATES, HH_AA_ESTIMATES,
    M12_14_AA_ESTIMATES, M15_17_AA_ESTIMATES, M18_20_AA_ESTIMATES,
    M18_34_AA_ESTIMATES, M18_49_AA_ESTIMATES, M18_AA_ESTIMATES,
    M21_24_AA_ESTIMATES, M21_AA_ESTIMATES, M25_29_AA_ESTIMATES,
    M25_54_AA_ESTIMATES, M25_64_AA_ESTIMATES, M25_AA_ESTIMATES,
    M2_5_AA_ESTIMATES, M2_AA_ESTIMATES, M30_34_AA_ESTIMATES,
    M35_39_AA_ESTIMATES, M40_44_AA_ESTIMATES, M45_49_AA_ESTIMATES,
    M50_54_AA_ESTIMATES, M55_64_AA_ESTIMATES, M65_AA_ESTIMATES,
    M6_8_AA_ESTIMATES, M9_11_AA_ESTIMATES, P18_34_AA_ESTIMATES,
    P18_49_AA_ESTIMATES, P18_AA_ESTIMATES, P21_AA_ESTIMATES,
    P25_54_AA_ESTIMATES, P25_64_AA_ESTIMATES, P25_AA_ESTIMATES,
    P2_17_AA_ESTIMATES, P2_AA_ESTIMATES, P35_54_AA_ESTIMATES,
    P50_AA_ESTIMATES, P55_AA_ESTIMATES, P65_AA_ESTIMATES,
    CASE NETWORK_CODE
        WHEN 'LIF'  THEN F25_64_AA_ESTIMATES
        WHEN 'LMN'  THEN F25_64_AA_ESTIMATES
        WHEN 'VICE' THEN P18_49_AA_ESTIMATES
        ELSE             P25_64_AA_ESTIMATES
    END                                                        AS DEMO_AA,
    SOURCE_FEED, VT AS VIEWING_TYPE,
    TRACKAGE_ID, FEED_PATTERN_IND, MARKET_BREAK_ID,
    FILE_NAME, FILE_DATE,
    CURRENT_TIMESTAMP()                                         AS LOADED_AT
FROM ranked
QUALIFY ROW_NUMBER() OVER (
        PARTITION BY NETWORK_CODE, RATING_SOURCE, BROADCAST_DATE,
                     PROGRAM_CODE, TELECAST_NUM, QTR_START_TIME
        ORDER BY FEED_RANK, WKLY_RANK, VT_RANK, TRK_RANK, MKT_RANK,
                 FILE_DATE DESC NULLS LAST, FILE_NAME DESC NULLS LAST
    ) = 1;
  ALTER TABLE AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT SWAP WITH AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT_NEW;
  DROP TABLE IF EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT_NEW;
  INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
    SELECT CURRENT_TIMESTAMP(), 'CRIS_QH_FACT', 'REBUILT', COUNT(*), MAX(BROADCAST_DATE), NULL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT;
END;

## 4. T4_LATEST_DATE -> CRIS_LATEST_DATE

In [ ]:
%%sql
CREATE OR REPLACE TASK AUDIENCE_DEV_DB.CRIS.T4_LATEST_DATE
  WAREHOUSE = EXT_SIIRA_XS_WH
  AFTER AUDIENCE_DEV_DB.CRIS.T3_QH_FACT
AS
BEGIN
  CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE_NEW AS
SELECT
    t.NETWORK_CODE,
    t.RATING_SOURCE,
    t.LATEST_BROADCAST_DATE,
    q.LATEST_QH_BROADCAST_DATE,
    DATEDIFF('day', q.LATEST_QH_BROADCAST_DATE, t.LATEST_BROADCAST_DATE) AS QH_LAG_DAYS,
    DATEADD('day', -MOD(DAYOFWEEKISO(t.LATEST_BROADCAST_DATE),7), t.LATEST_BROADCAST_DATE)
                                                          AS LATEST_FULL_SUNDAY,
    t.TELECAST_ROWS,
    CURRENT_TIMESTAMP()                                   AS REFRESHED_AT
FROM (
    SELECT NETWORK_CODE, RATING_SOURCE,
           MAX(BROADCAST_DATE) AS LATEST_BROADCAST_DATE,
           COUNT(*)            AS TELECAST_ROWS
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
    GROUP BY 1,2
) t
LEFT JOIN (
    SELECT NETWORK_CODE, RATING_SOURCE,
           MAX(BROADCAST_DATE) AS LATEST_QH_BROADCAST_DATE
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT
    GROUP BY 1,2
) q ON q.NETWORK_CODE = t.NETWORK_CODE AND q.RATING_SOURCE = t.RATING_SOURCE;
  ALTER TABLE AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE SWAP WITH AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE_NEW;
  DROP TABLE IF EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE_NEW;
  INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
    SELECT CURRENT_TIMESTAMP(), 'CRIS_LATEST_DATE', 'REBUILT', COUNT(*), MAX(BROADCAST_DATE), NULL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE;
END;

## 5. T5_NETWORK_UNIVERSE -> CRIS_NETWORK_UNIVERSE

In [ ]:
%%sql
CREATE OR REPLACE TASK AUDIENCE_DEV_DB.CRIS.T5_NETWORK_UNIVERSE
  WAREHOUSE = EXT_SIIRA_XS_WH
  AFTER AUDIENCE_DEV_DB.CRIS.T4_LATEST_DATE
AS
BEGIN
  CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE_NEW AS
WITH seen AS (
    SELECT NETWORK_CODE,
           MAX(MEDIA_TYPE_CODE)                                AS MEDIA_TYPE_CODE,
           MAX(COALESCE(TRY_TO_NUMBER(TO_VARCHAR(ORIGINATOR_ID)), -1)) AS ORIGINATOR_ID,
           COUNT(*)                                            AS HALFHRS,
           MIN(BROADCAST_DATE)                                 AS FROM_DATE,
           MAX(BROADCAST_DATE)                                 AS TO_DATE
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    GROUP BY NETWORK_CODE
),
ct AS (
    SELECT NETWORK_CODE,
           COUNT(*)            AS PREMIERE_HALFHRS,
           COUNT(CT_SEASON)    AS WITH_SEASON
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT
    WHERE REPEAT_IND     = 0
      AND RATING_SOURCE  = 'Live+SD'
      AND BROADCAST_DATE >= DATEADD('month', -12, CURRENT_DATE())
    GROUP BY NETWORK_CODE
)
SELECT
    s.NETWORK_CODE,
    s.MEDIA_TYPE_CODE,
    s.ORIGINATOR_ID,
    s.HALFHRS, s.FROM_DATE, s.TO_DATE,
    IFF(s.NETWORK_CODE IN ('AEN','HIST','LIF','LMN','FYI','VICE'), TRUE, FALSE)      AS IS_AE,
    IFF(s.MEDIA_TYPE_CODE = 'NHI', TRUE, FALSE)                AS IS_CABLE,
    IFF(s.MEDIA_TYPE_CODE <> 'NHI', TRUE, FALSE)               AS IS_BROADCAST,
    IFF(UPPER(s.NETWORK_CODE) = 'PV', TRUE, FALSE)             AS IS_ARTIFACT,
    IFF(s.MEDIA_TYPE_CODE = 'NHI'
        AND UPPER(s.NETWORK_CODE) <> 'PV'
        AND s.ORIGINATOR_ID NOT IN (
                  6521,6202,4964,8889,6204,7287,2263,7401,7443,9042,
                  6560,7564,7801,9772,9540,10680,6788,9897),
        TRUE, FALSE)                                           AS IS_ENT_BY_CODE,
    IFF(s.MEDIA_TYPE_CODE = 'NHI'
        AND UPPER(s.NETWORK_CODE) <> 'PV'
        AND UPPER(s.NETWORK_CODE) NOT IN (
                  'CNBC','CNN','ESDN','ESNU','ESPN','ESPN2','FBN','FOXN','FS1','FS2',
                  'GOLF','MLBN','MSNW','NBAT','NFLN','NMX','NWSN','TENN'),
        TRUE, FALSE)                                           AS IS_ENT_BY_NAME,
    c.PREMIERE_HALFHRS,
    ROUND(100.0 * c.WITH_SEASON / NULLIF(c.PREMIERE_HALFHRS, 0), 1)
                                                               AS CT_SEASON_PCT,
    IFF(COALESCE(c.PREMIERE_HALFHRS, 0) >= 200
        AND c.WITH_SEASON / NULLIF(c.PREMIERE_HALFHRS, 0) >= 0.90,
        TRUE, FALSE)                                           AS HAS_CT_COVERAGE,
    CURRENT_TIMESTAMP()                                        AS LOADED_AT
FROM seen s
LEFT JOIN ct c ON c.NETWORK_CODE = s.NETWORK_CODE;
  ALTER TABLE AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE SWAP WITH AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE_NEW;
  DROP TABLE IF EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE_NEW;
  INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
    SELECT CURRENT_TIMESTAMP(), 'CRIS_NETWORK_UNIVERSE', 'REBUILT', COUNT(*), NULL, NULL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_NETWORK_UNIVERSE;
END;

## 6. T6_FY_CALENDAR -> CRIS_FY_CALENDAR

In [ ]:
%%sql
CREATE OR REPLACE TASK AUDIENCE_DEV_DB.CRIS.T6_FY_CALENDAR
  WAREHOUSE = EXT_SIIRA_XS_WH
  AFTER AUDIENCE_DEV_DB.CRIS.T5_NETWORK_UNIVERSE
AS
BEGIN
  CREATE OR REPLACE TABLE AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR_NEW AS
WITH bounds AS (
    SELECT TRY_TO_NUMBER(FISCAL_YEAR)  AS FISCAL_YEAR,
           MIN(CALENDAR_DATE)          AS FY_START,
           MAX(CALENDAR_DATE)          AS FY_END
    FROM Reference_DB.MDM.Dim_DATE
    WHERE FISCAL_YEAR IS NOT NULL
    GROUP BY TRY_TO_NUMBER(FISCAL_YEAR)
)
SELECT
    FISCAL_YEAR,
    FY_START,
    FY_END,
    LAG(FY_START) OVER (ORDER BY FISCAL_YEAR)                  AS PRIOR_FY_START,
    LAG(FY_END)   OVER (ORDER BY FISCAL_YEAR)                  AS PRIOR_FY_END,
    CURRENT_TIMESTAMP()                                        AS LOADED_AT
FROM bounds
ORDER BY FISCAL_YEAR;
  ALTER TABLE AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR SWAP WITH AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR_NEW;
  DROP TABLE IF EXISTS AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR_NEW;
  INSERT INTO AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG (RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE)
    SELECT CURRENT_TIMESTAMP(), 'CRIS_FY_CALENDAR', 'REBUILT', COUNT(*), NULL, NULL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR;
END;

## 7. Enable the chain (children first, root last)

In [ ]:
%%sql
ALTER TASK AUDIENCE_DEV_DB.CRIS.T6_FY_CALENDAR RESUME;
ALTER TASK AUDIENCE_DEV_DB.CRIS.T5_NETWORK_UNIVERSE RESUME;
ALTER TASK AUDIENCE_DEV_DB.CRIS.T4_LATEST_DATE RESUME;
ALTER TASK AUDIENCE_DEV_DB.CRIS.T3_QH_FACT RESUME;
ALTER TASK AUDIENCE_DEV_DB.CRIS.T2_HALFHR_FACT RESUME;
ALTER TASK AUDIENCE_DEV_DB.CRIS.T1_TELECAST_FACT RESUME;

## 8. Run the chain once, now

In [ ]:
%%sql
EXECUTE TASK AUDIENCE_DEV_DB.CRIS.T1_TELECAST_FACT;

## 9. Watch it

In [ ]:
%%sql
SELECT NAME, STATE, SCHEDULED_TIME, QUERY_START_TIME, COMPLETED_TIME,
       DATEDIFF('minute', QUERY_START_TIME, COMPLETED_TIME) AS MINUTES, ERROR_MESSAGE
FROM TABLE(AUDIENCE_DEV_DB.INFORMATION_SCHEMA.TASK_HISTORY(
       SCHEDULED_TIME_RANGE_START => DATEADD('day', -3, CURRENT_TIMESTAMP()),
       RESULT_LIMIT => 100))
WHERE DATABASE_NAME = 'AUDIENCE_DEV_DB' AND SCHEMA_NAME = 'CRIS'
ORDER BY SCHEDULED_TIME DESC, NAME;

## 10. Are the tables level with their sources?

In [ ]:
%%sql
SELECT 'CRIS_TELECAST_FACT' AS TBL,
       (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT) AS TABLE_MAX,
       (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DB.UBD.VW_TOTAL_PROGRAM_ESTIMATES_UNIFIED) AS SOURCE_MAX
UNION ALL
SELECT 'CRIS_HALFHR_FACT',
       (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DEV_DB.CRIS.CRIS_HALFHR_FACT),
       (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DB.UBD.VW_HALFHR_PROGRAM_ESTIMATES_UNIFIED)
UNION ALL
SELECT 'CRIS_QH_FACT',
       (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT),
       (SELECT MAX(BROADCAST_DATE) FROM AUDIENCE_DB.UBD.VW_QTRHR_PROGRAM_ESTIMATES_UNIFIED)
UNION ALL
SELECT 'CRIS_LATEST_DATE', (SELECT MAX(REFRESHED_AT)::DATE FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE), CURRENT_DATE();

## Pause / remove

To stop the refresh: `ALTER TASK AUDIENCE_DEV_DB.CRIS.T1_TELECAST_FACT SUSPEND;` (suspending the root stops the chain). To remove: `DROP TASK` each one, children first.

## 11. What has been rebuilt, and how many rows

In [ ]:
%%sql
SELECT RUN_TS, TABLE_NAME, STATUS, ROW_COUNT, MAX_BROADCAST_DATE, NOTE
FROM AUDIENCE_DEV_DB.CRIS.CRIS_REFRESH_LOG
ORDER BY RUN_TS DESC
LIMIT 100;